# 05 — Temporal Stability and Final Date-Range Decision

This notebook compares the required DSNY Graffiti candidate periods, evaluates outcome maturity and possible regime changes, and makes the final Month 1 scope recommendation. It does not train a model or create data splits.

## 1. Imports, paths, and central decision rules

All network requests use the shared API client. Reusable metric and decision logic lives in `urban_ops.analysis.temporal_stability`.

In [1]:
from datetime import datetime, timezone
import os
from pathlib import Path
import sys
from urllib.parse import urlencode

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'urban_ops').is_dir())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.analysis.temporal_stability import (
    APPROVED_WITH_LIMITATIONS, DEFAULT_THRESHOLDS,
    build_monthly_scope_metrics, build_yearly_scope_metrics,
    evaluate_candidate_periods,
)
from urban_ops.data.api_client import fetch_json_url
from urban_ops.data.nyc_311_config import API_ENDPOINT, API_MAX_ATTEMPTS, API_TIMEOUT_SECONDS
from urban_ops.utils.paths import ensure_report_directories, notebook_report_paths

REPORT_DIR, TABLE_DIR, FIGURE_DIR = notebook_report_paths('05_temporal_stability')
_, SCOPE_TABLE_DIR, _ = notebook_report_paths('04_scope_selection')
ensure_report_directories('05_temporal_stability')
ensure_report_directories('04_scope_selection')
DOC_SCOPE_PATH = PROJECT_ROOT / 'docs' / 'scope_decision.md'
TARGET_DOC_PATH = PROJECT_ROOT / 'docs' / 'target_definition.md'
pd.set_option('display.max_columns', 60)
display(pd.DataFrame([DEFAULT_THRESHOLDS.to_dict()]).T.rename(columns={0: 'configured_value'}))

,configured_value
minimum_eligible_records,10000.00
minimum_due_date_coverage,0.70
minimum_closed_date_coverage,0.80
minimum_target_rate,0.05
maximum_target_rate,0.95
minimum_active_months,24.00
minimum_monthly_eligible_volume,250.00
minimum_complete_calendar_years,2.00
minimum_outcome_maturity_rate,0.98
maximum_invalid_timestamp_rate,0.05


## 2. Deterministic scoped extraction

The extraction is restricted to DSNY Graffiti complaints and ordered by `created_date, unique_key`. Pagination is deterministic. The extraction timestamp is captured before querying and is the single maturity cutoff used by every candidate.

In [2]:
AGENCY = 'DSNY'
AGENCY_NAME = 'Department of Sanitation'
COMPLAINT_TYPE = 'Graffiti'
QUERY_START = '2022-01-01T00:00:00.000'
QUERY_END_EXCLUSIVE = '2026-07-01T00:00:00.000'
PAGE_SIZE = 50000
COLUMNS = ['unique_key', 'created_date', 'closed_date', 'due_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'status']
extraction_timestamp = datetime.now(timezone.utc)
headers = {'User-Agent': 'urban-operations-intelligence-temporal-stability/1.0'}
if os.getenv('NYC_OPEN_DATA_APP_TOKEN'):
    headers['X-App-Token'] = os.environ['NYC_OPEN_DATA_APP_TOKEN']

def execute_soql(query: str) -> list[dict[str, object]]:
    """Execute one read-only query through the shared API client."""
    payload = fetch_json_url(
        f"{API_ENDPOINT}?{urlencode({'$query': query})}",
        timeout_seconds=API_TIMEOUT_SECONDS,
        max_attempts=API_MAX_ATTEMPTS,
        headers=headers,
    )
    if not isinstance(payload, list) or not all(isinstance(row, dict) for row in payload):
        raise RuntimeError('NYC Open Data returned an invalid record collection.')
    return payload

where = (
    "agency = 'DSNY' AND complaint_type = 'Graffiti' "
    f"AND created_date >= '{QUERY_START}' AND created_date < '{QUERY_END_EXCLUSIVE}'"
)
records = []
offset = 0
while True:
    query = (f"SELECT {', '.join(COLUMNS)} WHERE {where} "
             f"ORDER BY created_date, unique_key LIMIT {PAGE_SIZE} OFFSET {offset}")
    page = execute_soql(query)
    records.extend(page)
    if len(page) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

scope_records = pd.DataFrame.from_records(records).reindex(columns=COLUMNS)
if scope_records.empty:
    raise RuntimeError('The scoped NYC Open Data extraction is empty.')
missing_columns = sorted(set(COLUMNS).difference(scope_records.columns))
if missing_columns:
    raise RuntimeError(f'Scoped extraction is missing columns: {missing_columns}')
if not scope_records['agency'].eq(AGENCY).all() or not scope_records['complaint_type'].eq(COMPLAINT_TYPE).all():
    raise RuntimeError('The API response contains records outside the requested scope.')
extraction_metadata = pd.DataFrame([{
    'source': API_ENDPOINT, 'dataset_identifier': 'erm2-nwe9',
    'extraction_timestamp': extraction_timestamp.isoformat(),
    'agency': AGENCY, 'complaint_type': COMPLAINT_TYPE,
    'requested_start_date': '2022-01-01', 'requested_end_date': '2026-06-30',
    'row_count': len(scope_records), 'ordering': 'created_date ASC, unique_key ASC',
    'page_size': PAGE_SIZE, 'page_count': (len(scope_records) + PAGE_SIZE - 1) // PAGE_SIZE,
}])
extraction_metadata.to_csv(TABLE_DIR / 'extraction_metadata.csv', index=False)
display(extraction_metadata.T)

,0
source,https://data.cityofnewyork.us/resource/erm2-nw...
dataset_identifier,erm2-nwe9
extraction_timestamp,2026-07-27T16:11:48.990667+00:00
agency,DSNY
complaint_type,Graffiti
requested_start_date,2022-01-01
requested_end_date,2026-06-30
row_count,73851
ordering,"created_date ASC, unique_key ASC"
page_size,50000


## 3. Candidate periods and outcome maturity

`outcome_mature` requires a non-null due date no later than the extraction timestamp. The established target remains available only when creation, due, and closure timestamps are all present. Open complaints are measured but excluded from target construction; they are not silently treated as on time or late.

In [3]:
CANDIDATE_PERIODS = [
    ('2022-01-01 through 2026-06-30', '2022-01-01', '2026-06-30'),
    ('2022-01-01 through 2025-12-31', '2022-01-01', '2025-12-31'),
    ('2023-01-01 through 2026-06-30', '2023-01-01', '2026-06-30'),
    ('2023-01-01 through 2025-12-31', '2023-01-01', '2025-12-31'),
    ('2024-01-01 through 2026-06-30', '2024-01-01', '2026-06-30'),
    ('2024-01-01 through 2025-12-31', '2024-01-01', '2025-12-31'),
]
comparison, monthly_by_candidate, yearly_by_candidate = evaluate_candidate_periods(
    scope_records, CANDIDATE_PERIODS,
    extraction_timestamp=extraction_timestamp, thresholds=DEFAULT_THRESHOLDS,
)
comparison.to_csv(TABLE_DIR / 'temporal_stability_summary.csv', index=False)
comparison.to_csv(SCOPE_TABLE_DIR / 'date_range_candidate_comparison.csv', index=False)
thresholds_table = pd.DataFrame([{'threshold': key, 'value': value} for key, value in DEFAULT_THRESHOLDS.to_dict().items()])
thresholds_table.to_csv(TABLE_DIR / 'decision_thresholds.csv', index=False)
display(comparison[[
    'candidate_name', 'eligible_records', 'due_date_coverage', 'closed_date_coverage',
    'outcome_maturity_rate', 'missed_target_rate', 'missing_months',
    'monthly_target_rate_range', 'monthly_target_rate_standard_deviation',
    'largest_absolute_month_to_month_target_rate_change',
    'severe_temporal_volatility', 'decision_status', 'selected',
]].round(4))

,candidate_name,eligible_records,due_date_coverage,closed_date_coverage,outcome_maturity_rate,missed_target_rate,missing_months,monthly_target_rate_range,monthly_target_rate_standard_deviation,largest_absolute_month_to_month_target_rate_change,severe_temporal_volatility,decision_status,selected
0,2024-01-01 through 2025-12-31,35966,0.9055,0.9932,1.0000,0.4370,0,0.3023,0.0762,0.2156,False,APPROVED_WITH_LIMITATIONS,True
1,2023-01-01 through 2025-12-31,48427,0.9086,0.9946,1.0000,0.4374,0,0.4662,0.1066,0.2156,False,APPROVED_WITH_LIMITATIONS,False
2,2024-01-01 through 2026-06-30,41975,0.9082,0.9807,0.9986,0.4248,0,0.4593,0.0867,0.2165,False,APPROVED_WITH_LIMITATIONS,False
3,2022-01-01 through 2025-12-31,60281,0.9108,0.9944,1.0000,0.5335,0,0.7236,0.2427,0.3582,True,REQUIRES_NARROWER_RANGE,False
4,2023-01-01 through 2026-06-30,54436,0.9104,0.9847,0.9989,0.4280,0,0.5904,0.1101,0.2165,True,REQUIRES_NARROWER_RANGE,False
5,2022-01-01 through 2026-06-30,66290,0.9120,0.9862,0.9991,0.5170,0,0.8478,0.2413,0.3582,True,REQUIRES_NARROWER_RANGE,False


## 4. Final selection and period-consistent outputs

Feasibility is evaluated before ranking. Severe volatile ranges are required to narrow when a feasible, materially sized stable subset exists. Among feasible non-severe candidates ending at the latest complete calendar-year boundary, the score balances target-rate stability, volume stability, usable volume, and recent relevance.

In [4]:
selected = comparison.loc[comparison['selected']].iloc[0]
selected_name = str(selected['candidate_name'])
selected_monthly = monthly_by_candidate[selected_name].copy()
selected_yearly = yearly_by_candidate[selected_name].copy()
selected_monthly.to_csv(TABLE_DIR / 'selected_scope_monthly_metrics.csv', index=False)
selected_yearly.to_csv(TABLE_DIR / 'selected_scope_yearly_metrics.csv', index=False)
selected_monthly.to_csv(SCOPE_TABLE_DIR / 'selected_scope_monthly_metrics.csv', index=False)
selected_yearly.to_csv(SCOPE_TABLE_DIR / 'selected_scope_yearly_metrics.csv', index=False)
yearly_all = pd.concat([frame.assign(candidate_name=name) for name, frame in yearly_by_candidate.items()], ignore_index=True)
yearly_all.to_csv(TABLE_DIR / 'candidate_yearly_metrics.csv', index=False)
selected_summary = pd.DataFrame([{
    'decision_status': APPROVED_WITH_LIMITATIONS, 'selected_agency': AGENCY,
    'selected_agency_name': AGENCY_NAME, 'selected_complaint_type': COMPLAINT_TYPE,
    'selected_start_date': selected['start_date'], 'selected_end_date': selected['end_date'],
    'extraction_timestamp': extraction_timestamp.isoformat(),
    **{column: selected[column] for column in comparison.columns if column not in {'selected_agency'}},
}])
selected_summary.to_csv(TABLE_DIR / 'selected_scope_summary.csv', index=False)
selected_summary.to_csv(SCOPE_TABLE_DIR / 'selected_scope_summary.csv', index=False)
complaint_evidence = pd.DataFrame([{
    'agency': AGENCY, 'agency_name': AGENCY_NAME, 'complaint_type': COMPLAINT_TYPE,
    'evidence_start_date': '2022-01-01', 'evidence_end_date': '2026-06-30',
    'inclusion_reason': 'Complaint subgroup passes volume, coverage, class-balance, continuity, and timestamp gates; date range is decided separately.',
    'final_scope_start_date': selected['start_date'], 'final_scope_end_date': selected['end_date'],
}])
complaint_evidence.to_csv(SCOPE_TABLE_DIR / 'complaint_type_inclusion_evidence.csv', index=False)
rejected = comparison.loc[~comparison['selected']].copy()
rejected['rejection_reason'] = rejected.apply(lambda row: (
    row['decision_reasons'] or ('Includes incomplete 2026 and weaker outcome maturity/closure coverage' if row['end_date'] == '2026-06-30' else 'Less stable than the selected recent complete-calendar period')
), axis=1)
rejected.to_csv(SCOPE_TABLE_DIR / 'rejected_scope_candidates.csv', index=False)
display(selected_summary.T)

,0
decision_status,APPROVED_WITH_LIMITATIONS
selected_agency,DSNY
selected_agency_name,Department of Sanitation
selected_complaint_type,Graffiti
selected_start_date,2024-01-01
selected_end_date,2025-12-31
extraction_timestamp,2026-07-27T16:11:48.990667+00:00
candidate_name,2024-01-01 through 2025-12-31
start_date,2024-01-01
end_date,2025-12-31


## 5. Yearly and regime-change evidence

Abrupt monthly changes, sustained yearly differences, coverage changes, volume changes, and descriptor composition changes are diagnostic signals. They do not establish a causal explanation.

In [5]:
full_name = CANDIDATE_PERIODS[0][0]
full_monthly = monthly_by_candidate[full_name].copy()
full_monthly['absolute_target_rate_change'] = full_monthly['missed_target_rate'].diff().abs()
full_monthly['eligible_volume_change_rate'] = full_monthly['eligible_records'].pct_change().replace([float('inf'), float('-inf')], pd.NA)
full_monthly['due_coverage_change'] = full_monthly['due_date_coverage'].diff()
abrupt = full_monthly.nlargest(10, 'absolute_target_rate_change')[[
    'month', 'eligible_records', 'missed_target_rate', 'absolute_target_rate_change',
    'due_date_coverage', 'due_coverage_change', 'eligible_volume_change_rate',
]].copy()
abrupt['interpretation'] = 'Possible operational or data-generation regime boundary; cause not established by this dataset.'
abrupt.to_csv(TABLE_DIR / 'regime_change_evidence.csv', index=False)

descriptor_work = scope_records.copy()
descriptor_work['created_date'] = pd.to_datetime(descriptor_work['created_date'], errors='coerce', utc=True)
descriptor_work['year'] = descriptor_work['created_date'].dt.year
descriptor_work['descriptor_label'] = descriptor_work['descriptor'].fillna('<MISSING>')
descriptor_yearly = descriptor_work.groupby(['year', 'descriptor_label'], dropna=False).size().rename('record_count').reset_index()
descriptor_totals = descriptor_yearly.groupby('year')['record_count'].transform('sum')
descriptor_yearly['year_share'] = descriptor_yearly['record_count'] / descriptor_totals
descriptor_yearly = descriptor_yearly.sort_values(['year', 'record_count'], ascending=[True, False])
descriptor_yearly.to_csv(TABLE_DIR / 'descriptor_yearly_distribution.csv', index=False)
display(Markdown('**Regime interpretation.** The data indicates a possible operational or data-generation regime change. The available dataset does not establish the underlying cause.'))
display(abrupt.head())
display(yearly_by_candidate[full_name])

**Regime interpretation.** The data indicates a possible operational or data-generation regime change. The available dataset does not establish the underlying cause.

,month,eligible_records,missed_target_rate,absolute_target_rate_change,due_date_coverage,due_coverage_change,eligible_volume_change_rate,interpretation
12,2023-01-01,429,0.631702,0.358173,0.902083,-0.035694,-0.744491,Possible operational or data-generation regime...
53,2026-06-01,1108,0.150722,0.216510,0.931174,0.033231,0.043315,Possible operational or data-generation regime...
35,2024-12-01,2367,0.552176,0.215554,0.899925,-0.026445,0.459309,Possible operational or data-generation regime...
1,2022-02-01,3692,0.812839,0.170633,0.964008,0.153594,5.102479,Possible operational or data-generation regime...
16,2023-05-01,444,0.463964,0.157123,0.886454,0.021206,-0.268534,Possible operational or data-generation regime...


,year,total_records,yearly_eligible_volume,due_date_present_count,closed_date_present_count,missed_count,yearly_due_date_coverage,yearly_closed_date_coverage,yearly_missed_target_rate
0,2022,12979,11854,11939,12892,10975,0.919871,0.993297,0.925848
1,2023,13595,12461,12477,13578,5466,0.917764,0.998750,0.438649
2,2024,21694,19574,19622,21646,8636,0.904490,0.997787,0.441198
3,2025,18323,16392,16614,18100,7080,0.906729,0.987830,0.431918
4,2026,7260,6009,6703,6618,2116,0.923278,0.911570,0.352138


## 6. Saved figures

All decision charts are written to `reports/05_temporal_stability/figures/`; notebook rendering is not the only copy.

In [6]:
def save_line(frame: pd.DataFrame, column: str, filename: str, title: str, *, percentage: bool = False) -> None:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(frame['month'], frame[column], linewidth=1.8, color='#4472C4')
    ax.axvspan(pd.Timestamp('2026-01-01'), pd.Timestamp('2026-06-30'), color='#F4B183', alpha=0.22, label='Incomplete 2026 calendar year')
    ax.set_title(title)
    ax.grid(alpha=0.25)
    if percentage:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_ylim(0, 1)
    ax.legend(loc='best')
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160, bbox_inches='tight')
    plt.close(fig)

save_line(full_monthly, 'eligible_records', 'monthly_eligible_volume.png', 'Monthly eligible DSNY Graffiti volume')
save_line(full_monthly, 'missed_target_rate', 'monthly_missed_target_rate.png', 'Monthly missed-target rate', percentage=True)
save_line(full_monthly, 'due_date_coverage', 'monthly_due_date_coverage.png', 'Monthly due-date coverage', percentage=True)

fig, ax = plt.subplots(figsize=(9, 5))
yearly_full = yearly_by_candidate[full_name]
ax.bar(yearly_full['year'].astype(str), yearly_full['yearly_missed_target_rate'], color='#4472C4')
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_ylim(0, 1)
ax.set_title('Yearly missed-target rate — DSNY Graffiti'); ax.grid(axis='y', alpha=0.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'yearly_missed_target_rate.png', dpi=160, bbox_inches='tight'); plt.close(fig)

plot = comparison.sort_values(['start_date', 'end_date']).copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#70AD47' if value else '#A5A5A5' for value in plot['selected']]
axes[0].barh(plot['candidate_name'], plot['monthly_target_rate_range'], color=colors)
axes[0].axvline(DEFAULT_THRESHOLDS.severe_target_rate_range, color='#C00000', linestyle='--', label='Severe threshold')
axes[0].set_title('Monthly target-rate range'); axes[0].xaxis.set_major_formatter(PercentFormatter(1.0)); axes[0].legend()
axes[1].barh(plot['candidate_name'], plot['eligible_records'], color=colors)
axes[1].set_title('Outcome-mature eligible records')
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'candidate_period_comparison.png', dpi=160, bbox_inches='tight'); plt.close(fig)
required_figures = ['monthly_eligible_volume.png', 'monthly_missed_target_rate.png', 'monthly_due_date_coverage.png', 'yearly_missed_target_rate.png', 'candidate_period_comparison.png']
assert all((FIGURE_DIR / name).is_file() for name in required_figures)

## 7. Explicit final recommendation

The selected period is **2024-01-01 through 2025-12-31**. It retains two complete, recent calendar years and substantial representation of both target classes while materially reducing target-rate and volume instability. Longer periods are rejected because early 2022 and the incomplete 2026 calendar year introduce substantially different target behaviour; the 2026 tail also has weaker closure completeness. The 2023–2025 option provides more records but remains more volatile. No shorter required option is supported because fewer than two complete calendar years would weaken seasonal representation.

Residual risk remains: even the selected period crosses the configured warning threshold, so the final status is **APPROVED_WITH_LIMITATIONS**. Downstream work must use chronological evaluation, report metrics by month/year, test drift, and must not assume stationarity. Outcome maturity changes the end-date decision in the sense that 2026 is not a complete calendar year and its recent closure observations are less complete; no due date after the extraction timestamp is allowed into eligible evidence.

In [7]:
assert selected['start_date'] == '2024-01-01'
assert selected['end_date'] == '2025-12-31'
assert selected['decision_status'] in {'APPROVED', 'APPROVED_WITH_LIMITATIONS'}
assert selected['missing_months'] == 0
assert selected['eligible_records'] >= DEFAULT_THRESHOLDS.minimum_eligible_records
assert selected['missed_count'] > 0 and selected['on_time_count'] > 0
assert set(selected_monthly['month'].dt.year) == {2024, 2025}
assert all((TABLE_DIR / name).is_file() for name in [
    'temporal_stability_summary.csv', 'selected_scope_summary.csv',
    'selected_scope_monthly_metrics.csv', 'selected_scope_yearly_metrics.csv',
    'regime_change_evidence.csv', 'descriptor_yearly_distribution.csv',
])
display(Markdown(f"**Final decision: APPROVED_WITH_LIMITATIONS — DSNY Graffiti, {selected['start_date']} through {selected['end_date']}.**"))

**Final decision: APPROVED_WITH_LIMITATIONS — DSNY Graffiti, 2024-01-01 through 2025-12-31.**